# Baum-Welch Training

The Baum-Welch algorithm is an expectation-maximization (EM) algorithm for training HMMs. It finds the model parameters $\lambda = (\mathbf{A}, \mathbf{B}, \boldsymbol{\pi})$ that maximize the likelihood of observed sequences.

## Mathematical Background

### Forward Variable

The **forward variable** $\alpha_t(i)$ is the probability of being in state $S_i$ at time $t$ after observing the first $t$ observations:


$\alpha_t(i) = P(q_t = S_i, o_1, ..., o_t | \lambda)$


### Backward Variable

The **backward variable** $\beta_t(i)$ is the probability of observing the remaining observations from time $t+1$ to $T$ given we're in state $S_i$ at time $t$:

$\beta_t(i) = P(o_{t+1}, ..., o_T | q_t = S_i, \lambda)$

### Initialization (Rabiner Eq. 31)

$\beta_T(i) = 1 \quad \text{for all } i$

### Induction (Rabiner Eq. 31)

$\beta_t(i) = \sum_{j=1}^{N} a_{ij} b_j(o_{t+1}) \beta_{t+1}(j)$


### Reestimation Formulas (Rabiner Eq. 39-45)

**State occupancy probability**:

$\xi_t(i, j) = \frac{\alpha_t(i) a_{ij} b_j(o_{t+1}) \beta_{t+1}(j)}{P(O|\lambda)}$

**Reestimated transition matrix**:

$\bar{a}_{ij} = \frac{\sum_{t=1}^{T-1} \xi_t(i, j)}{\sum_{t=1}^{T-1} \sum_{k=1}^{N} \xi_t(i, k)}$

**Reestimated emission matrix**:

$\bar{b}_j(k) = \frac{\sum_{t=1}^{T} \gamma_t(j) \delta(o_t, v_k)}{\sum_{t=1}^{T} \gamma_t(j)}$

where $\gamma_t(j) = \sum_{i=1}^{N} \xi_t(i, j)$ is the probability of being in state $j$ at time $t$.

**Reestimated initial distribution**:

$\bar{\pi}_i = \gamma_1(i)$

The algorithm iteratively applies these reestimation formulas until convergence.

In [ ]:
from hmm import HMM, baum_welch, forward
from hmm.algorithms import ComputeMode


## Basic Training

In [ ]:
# Create initial HMM with random parameters
V = [0, 1, 2]
hmm_initial = HMM(n_states=2, V=V)

# Training data: sequences we want to learn
obs_seqs = [
    [0, 0, 0, 0, 1, 1],  # Mostly 0s, some 1s
    [0, 0, 0, 1, 1, 2],
    [0, 0, 1, 1, 1, 2],
    [0, 1, 1, 2, 2, 2],
]

# Check initial probability
initial_ll = sum(forward(hmm_initial, obs, mode=ComputeMode.SCALED)[0] for obs in obs_seqs)
print(f"Initial log-likelihood: {initial_ll:.4f}")

# Train the HMM
hmm_trained = baum_welch(
    hmm_initial,
    obs_seqs=obs_seqs,
    epochs=20,
    mode=ComputeMode.SCALED,
    verbose=True
)

## Training Results

In [ ]:
# Check final probability
final_ll = sum(forward(hmm_trained, obs, mode=ComputeMode.SCALED)[0] for obs in obs_seqs)
print(f"Final log-likelihood: {final_ll:.4f}")
print(f"Improvement: {final_ll - initial_ll:.4f}")

print("\nLearned transition matrix:")
print(hmm_trained.A)
print("\nLearned emission matrix:")
print(hmm_trained.B)
print("\nInitial state distribution:")
print(hmm_trained.Pi)

## With Validation Set

Use a validation set to prevent overfitting.

In [ ]:
# Training and validation sets
train_seqs = [[0, 0, 0, 1, 1], [0, 0, 1, 1, 2], [0, 1, 1, 2, 2]]
val_seqs = [[0, 0, 0, 0, 1], [0, 0, 1, 2, 2]]

# Train with validation
hmm_val = HMM(n_states=2, V=V)
hmm_best = baum_welch(
    hmm_val,
    obs_seqs=train_seqs,
    val_set=val_seqs,
    epochs=30,
    mode=ComputeMode.SCALED,
    verbose=True
)

## Selective Updates

You can choose which parameters to update.

In [ ]:
# Only update emission probabilities, keep transitions fixed
hmm_fixed = HMM(n_states=2, V=V)
original_a = hmm_fixed.A.copy()

hmm_trained = baum_welch(
    hmm_fixed,
    obs_seqs=obs_seqs,
    epochs=10,
    update_a=False,  # Keep transitions fixed
    mode=ComputeMode.SCALED
)

print("Original transition matrix (unchanged):")
print(original_a)
print("\nFinal transition matrix:")
print(hmm_trained.A)
print("\nEmission matrix (updated):")
print(hmm_trained.B)

## Summary

- Baum-Welch is an EM algorithm for HMM training
- Increases likelihood of observation sequences
- Can use validation set for early stopping
- Can selectively update A, B, or π

Next notebook: [Applications](./05-applications.ipynb)